# Biased Matrix Factorization 

In this notebook, we create from scratch a BMF algorithm, train it and validate it on the given dataset. The target is to port later the validated algorithm into our application backend for our users.  

Since the training dataset is not matching our database, we need this notebook to develop correctly our algorithm and then plug it into our project.

---

### The Algorithm

For a user $u$ and an event $i$, the predicted rating is:

$$\hat{r}_{ui} = \mu + b_u + b_i + \mathbf{p}_u^\top \mathbf{q}_i$$

- $\mu$ — global mean rating
- $b_u$ — user bias (how positive/negative this user tends to be)
- $b_i$ — item bias (how popular this event is overall)
- $\mathbf{p}_u \in \mathbb{R}^k$ — user latent vector
- $\mathbf{q}_i \in \mathbb{R}^k$ — item latent vector

We minimise the **regularised squared error** over observed ratings:

$$\min_{b_*, \mathbf{p}_*, \mathbf{q}_*} \sum_{(u,i)\in\mathcal{K}} \left(r_{ui} - \hat{r}_{ui}\right)^2
+ \lambda\left(b_u^2 + b_i^2 + \lVert\mathbf{p}_u\rVert^2 + \lVert\mathbf{q}_i\rVert^2\right)$$

trained with **stochastic gradient descent**.

## Set the environment

In [14]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RNG_SEED = 50
np.random.seed(RNG_SEED)

# point to the folder that holds the CSV files <<<
DATA_DIR = r"C:\Users\oiko\Desktop\DIT\Web_Applications\web_app\dataset\rel_event_csvs"

import os
print("Files in dataset folder:")
for f in os.listdir(DATA_DIR):
    size_mb = os.path.getsize(os.path.join(DATA_DIR, f)) / 1e6
    print(f"  {f:<25} {size_mb:8.1f} MB")

Files in dataset folder:
  events.csv                   641.6 MB
  event_attendees.csv          407.2 MB
  event_interest.csv             0.6 MB
  users.csv                      2.6 MB
  user_friends.csv             466.8 MB


## Load the training Dataset

In [15]:
event_interest = pd.read_csv(os.path.join(DATA_DIR, "event_interest.csv"))
print("Shape: ", event_interest.shape)
event_interest.head()

Shape:  (14978, 6)


,user,event,invited,timestamp,interested,not_interested
0,8949,1130067.0,0,2012-10-02 15:53:05.754,0,0
1,8949,1130381.0,0,2012-10-02 15:53:05.754,0,0
2,8949,1566551.0,0,2012-10-02 15:53:05.754,1,0
3,8949,1184649.0,0,2012-10-02 15:53:05.754,0,0
4,8949,1182052.0,0,2012-10-02 15:53:05.754,0,0


## Ratings

|           Signal         |      Rating         |
| :----------------------: | :-----------------: |
| `interested = 1`         |  **1.0** (positive) |
| `not_interested = 1`     |  **0.0** (negative) |
| both `0`                 |    *dropped*        |

In [16]:
def build_ratings(df):
    rows = []
    for user, event, interested, not_interested in zip(
        df["user"], df["event"], df["interested"], df["not_interested"]
    ):
        if interested == 1:
            rows.append((user, event, 1.0))
        elif not_interested == 1:
            rows.append((user, event, 0.0))
        # both 0 -> no real opinion
    return pd.DataFrame(rows, columns=["user", "event", "rating"])

ratings_interest = build_ratings(event_interest)
print("explicit ratings:", len(ratings_interest))
print("unique users :", ratings_interest["user"].nunique())
print("unique events:", ratings_interest["event"].nunique())
print("positive ratio:", round((ratings_interest.rating == 1.0).mean(), 3))
ratings_interest.head()

explicit ratings: 4547
unique users : 1979
unique events: 2739
positive ratio: 0.888


,user,event,rating
0,8949,1566551.0,1.0
1,20546,1806207.0,1.0
2,20546,1709580.0,1.0
3,20546,1751156.0,1.0
4,23668,NaN,1.0


## Enrich Ratings with Attendance  

The attendees of each event strongly show the preference as well. So it is only fair to add them in the total ratings to help our algorithm.

In [17]:
USE_ATTENDEES = True

if USE_ATTENDEES:
    status_to_rating = {"yes": 1.0, "maybe": 0.6, "no": 0.0}
    extra = []
    path = os.path.join(DATA_DIR, "event_attendees.csv")
    for chunk in pd.read_csv(path, usecols=["event", "status", "user_id"], chunksize=500_000):
        chunk = chunk.dropna(subset=["user_id"])
        chunk = chunk[chunk["status"].isin(status_to_rating)]
        for u, e, s in zip(chunk["user_id"], chunk["event"], chunk["status"]):
            extra.append((int(u), int(e), status_to_rating[s]))
    extra_df = pd.DataFrame(extra, columns=["user", "event", "rating"])
    print("attendance ratings:", len(extra_df))
    # combine; if a (user,event) appears in both, keep the explicit one (interest)
    ratings_full = pd.concat([ratings_interest, extra_df]).drop_duplicates(
        subset=["user", "event"], keep="first"
    ).reset_index(drop=True)
    print("combined ratings:", len(ratings_full))

attendance ratings: 10392
combined ratings: 14858


In [18]:
ratings_full[ratings_full['rating']==0.6].head()

,user,event,rating
9443,12769,2202380.0,0.6
9444,12948,2202380.0,0.6
9445,25277,1719406.0,0.6
9446,34911,1926768.0,0.6
9447,28115,1926768.0,0.6


## Clear Dataset

- Remap users' IDs to a continuous list `0,1,...,n-1`
- Drop users with very few interactions. It does not help the algorithm to learn, basically it is just noise
- Drop events with few interactions too.

In [70]:
MIN_USER_RATINGS = 3
MIN_EVENT_RATINGS = 3


# drop first the users with very few interactions, so after an event that he attented or showed interest can be dropped too.
prev = -1
while len(ratings_full) != prev:
    prev = len(ratings_full)
    uc = ratings_full["user"].value_counts()
    ratings_full = ratings_full[ratings_full["user"].isin(uc[uc >= MIN_USER_RATINGS].index)]
    ec = ratings_full["event"].value_counts()
    ratings_full = ratings_full[ratings_full["event"].isin(ec[ec >= MIN_EVENT_RATINGS].index)]

ratings_clean = ratings_full.reset_index(drop=True)

user_ids = ratings_clean["user"].unique()
event_ids = ratings_clean["event"].unique()
user2idx = {u: k for k, u in enumerate(user_ids)}
event2idx = {e: k for k, e in enumerate(event_ids)}
idx2event = {k: e for e, k in event2idx.items()}

n_users = len(user2idx)
n_items = len(event2idx)

triples = [
    (user2idx[u], event2idx[e], float(r))
    for u, e, r in zip(ratings_clean["user"], ratings_clean["event"], ratings_clean["rating"])
]
print(f"after filtering: {len(triples)} ratings, {n_users} users, {n_items} events")
print(f"matrix density: {len(triples) / (n_users * n_items):.4%}")

after filtering: 1689 ratings, 127 users, 217 events
matrix density: 6.1287%


In [71]:
# Final ratings
ratings = ratings_clean
ratings.head(10)

,user,event,rating
0,7627,1182052.0,1.0
1,7627,1566551.0,1.0
2,7627,1184938.0,1.0
3,7627,2352536.0,1.0
4,7627,2352476.0,1.0
5,3138,994197.0,1.0
6,3138,974595.0,1.0
7,3138,946591.0,1.0
8,3138,974483.0,1.0
9,10692,1566551.0,1.0


## Train / Test splits

Random 80/20 split for training and testing.

In [72]:
rng = np.random.default_rng(RNG_SEED)
perm = rng.permutation(len(triples))
split = int(0.8 * len(triples))
train = [triples[i] for i in perm[:split]]
test  = [triples[i] for i in perm[split:]]
print(f"train={len(train)}  test={len(test)}")

train=1351  test=338


## Biased Matrix Factorization Model

In [2]:
class BMF:
    """Biased Matrix Factorization trained with SGD.

        r_hat(u, i) = mu + b_u[u] + b_i[i] + P[u] . Q[i]
    """

    def __init__(self, factors=20, epochs=40, lr=0.01, reg=0.05, seed=42, verbose=True):
        self.factors = factors
        self.epochs = epochs
        self.lr = lr
        self.reg = reg
        self.seed = seed
        self.verbose = verbose

    def fit(self, train, users, items, val=None):
        rng = np.random.default_rng(self.seed)
        self.users, self.items = users, items
        self.mu = float(np.mean([r for _,_,r in train]))
        self.b_u = np.zeros(users)
        self.b_i = np.zeros(items)
        self.P = rng.normal(0, 0.1, (users, self.factors))
        self.Q = rng.normal(0, 0.1, (items, self.factors))

        train_arr = np.array(train, dtype=np.float64)
        self.history = {"train_rmse": [], "val_rmse": []}

        for epoch in range(self.epochs):
            rng.shuffle(train_arr)
            for u, i, r in train_arr:
                u, i = int(u), int(i)
                pred = self.mu + self.b_u[u] + self.b_i[i] + self.P[u] @ self.Q[i]
                e = r - pred
                self.b_u[u] += self.lr * (e - self.reg * self.b_u[u])
                self.b_i[i] += self.lr * (e - self.reg * self.b_i[i])
                p_u_old = self.P[u].copy()
                self.P[u] += self.lr * (e * self.Q[i] - self.reg * self.P[u])
                self.Q[i] += self.lr * (e * p_u_old - self.reg * self.Q[i])

            errs = [(r - self.predict(int(u), int(i))) ** 2 for u, i, r in train]
            training = float(np.sqrt(np.mean(errs)))
            self.history["train_rmse"].append(training)
            line = f"epoch {epoch+1:3d}  train_rmse={training:.4f}"
            if val is not None:
                errs = [(r - self.predict(int(u), int(i))) ** 2 for u, i, r in val]
                validation = float(np.sqrt(np.mean(errs)))
                self.history["val_rmse"].append(validation)
                line += f"  val_rmse={validation:.4f}"
            if self.verbose:
                print(line)
        return self
    
    def predict(self, u, i):
        return self.mu + self.b_u[u] + self.b_i[i] + self.P[u] @ self.Q[i]


    """Top-n events for user u, excluding already-seen items."""
    def recommend(self, u, seen_items, n=10):
        scores = self.mu + self.b_u[u] + self.b_i + self.Q @ self.P[u]
        scores[list(seen_items)] = -np.inf
        top = np.argpartition(-scores, min(n, len(scores) - 1))[:n]
        top = top[np.argsort(-scores[top])]
        return [(int(i), float(scores[i])) for i in top]


### Training

In [74]:
# ===================== K-fold cross-validation =====================

from collections import defaultdict

def kfold_cv(triples, n_users, n_items, K_FOLDS=5,
             factors=20, epochs=40, lr=0.01, reg=0.05,
             seed=42, K_REC=10, verbose=True):
    rng = np.random.default_rng(seed)
    perm = rng.permutation(len(triples))
    fold_size = len(triples) // K_FOLDS
    folds = [perm[k*fold_size:(k+1)*fold_size] for k in range(K_FOLDS)]
    if K_FOLDS * fold_size < len(triples):
        folds[-1] = np.concatenate([folds[-1], perm[K_FOLDS*fold_size:]])

    rows = []
    for k in range(K_FOLDS):
        val_idx   = folds[k]
        train_idx = np.concatenate([folds[j] for j in range(K_FOLDS) if j != k])
        train_k = [triples[i] for i in train_idx]
        val_k   = [triples[i] for i in val_idx]
        if verbose:
            print(f"\n--- Fold {k+1}/{K_FOLDS}  train={len(train_k)}  val={len(val_k)} ---")

        m = BMF(factors=factors, epochs=epochs, lr=lr, reg=reg,
                seed=seed + k, verbose=False)
        m.fit(train_k, n_users, n_items, val=val_k)

        # ----- RMSE metrics -----
        final_train_rmse = m.history["train_rmse"][-1]
        final_val_rmse   = m.history["val_rmse"][-1]
        best_val_rmse    = float(np.min(m.history["val_rmse"]))
        best_epoch       = int(np.argmin(m.history["val_rmse"])) + 1
        mu_k = np.mean([r for _, _, r in train_k])
        baseline_rmse = float(np.sqrt(np.mean([(r - mu_k)**2 for _, _, r in val_k])))

        # ----- Ranking eval setup -----
        train_by_user_k = defaultdict(set)
        for u, i, _ in train_k:
            train_by_user_k[int(u)].add(int(i))
        val_pos_by_user_k = defaultdict(set)
        for u, i, r in val_k:
            if r >= 1.0:
                val_pos_by_user_k[int(u)].add(int(i))
        eval_users_k = [u for u in val_pos_by_user_k if len(train_by_user_k[u]) > 0]
        n_eval = len(eval_users_k)

        # ----- BMF ranking -----
        hits, total = 0, 0
        for u in eval_users_k:
            recs = {i for i, _ in m.recommend(u, train_by_user_k[u], n=K_REC)}
            hits  += len(recs & val_pos_by_user_k[u])
            total += min(K_REC, len(val_pos_by_user_k[u]))
        precision = hits / (n_eval * K_REC) if n_eval > 0 else 0.0
        recall    = hits / total if total > 0 else 0.0

        # ----- Random-baseline ranking (deterministic per fold) -----
        rng_rand = np.random.default_rng(seed + 1000 + k)
        all_items = np.arange(n_items)
        rand_hits, rand_total = 0, 0
        for u in eval_users_k:
            candidates = all_items[~np.isin(all_items, list(train_by_user_k[u]))]
            pick = min(K_REC, len(candidates))
            rand_recs = set(rng_rand.choice(candidates, size=pick, replace=False).tolist())
            rand_hits  += len(rand_recs & val_pos_by_user_k[u])
            rand_total += min(K_REC, len(val_pos_by_user_k[u]))
        rand_precision = rand_hits / (n_eval * K_REC) if n_eval > 0 else 0.0
        rand_recall    = rand_hits / rand_total if rand_total > 0 else 0.0

        row = {
            "fold": k + 1,
            "train_rmse": final_train_rmse,
            "val_rmse": final_val_rmse,
            "best_val_rmse": best_val_rmse,
            "best_epoch": best_epoch,
            "baseline_rmse": baseline_rmse,
            "improvement_pct": (1 - final_val_rmse / baseline_rmse) * 100,
            f"precision@{K_REC}": precision,
            f"random_p@{K_REC}": rand_precision,
            f"lift_p@{K_REC}": precision - rand_precision,
            f"recall@{K_REC}": recall,
            f"random_r@{K_REC}": rand_recall,
            "n_eval_users": n_eval,
        }
        rows.append(row)
        if verbose:
            print(f"  train_rmse={final_train_rmse:.4f}  val_rmse={final_val_rmse:.4f}  "
                  f"best_val={best_val_rmse:.4f} @epoch {best_epoch}")
            print(f"  baseline={baseline_rmse:.4f}  improvement={row['improvement_pct']:.1f}%")
            print(f"  P@{K_REC}={precision:.4f}  (random={rand_precision:.4f}  lift={precision-rand_precision:+.4f})")
            print(f"  R@{K_REC}={recall:.4f}  (random={rand_recall:.4f})  n_eval={n_eval}")

    return pd.DataFrame(rows)

def summarize_cv(df_metrics):
    print("\n========== K-Fold CV Summary ==========")
    print(df_metrics.to_string(index=False))
    print("\nMean ± Std across folds:")
    for col in df_metrics.columns:
        if col in ("fold", "best_epoch", "n_eval_users"):
            continue
        v = df_metrics[col]
        print(f"  {col:18s}: {v.mean():.4f} ± {v.std():.4f}")

In [80]:

results = []

for factors in [3, 4]:
    for lr in [0.02, 0.025, 0.03, 0.04, 0.05]:
        for reg in [0.01, 0.02, 0.03, 0.05]:
            df_cv = kfold_cv(triples, n_users, n_items,
                            K_FOLDS=5,
                            factors=factors, epochs=60, lr=lr, reg=reg,
                            seed=RNG_SEED, verbose=True)
            summarize_cv(df_cv)
            print("="*100)
            results.append({
                "factors": factors, "lr": lr, "reg": reg, "epochs": 60,
                "mean_p@10":            df_cv["precision@10"].mean(),
                "std_p@10":             df_cv["precision@10"].std(),
                "mean_lift_p@10":       df_cv["lift_p@10"].mean(),
                "mean_r@10":            df_cv["recall@10"].mean(),
                "mean_val_rmse":        df_cv["val_rmse"].mean(),
                "mean_improvement_pct": df_cv["improvement_pct"].mean(),
            })


--- Fold 1/5  train=1352  val=337 ---
  train_rmse=0.2088  val_rmse=0.3806  best_val=0.3607 @epoch 10
  baseline=0.4439  improvement=14.3%
  P@10=0.0158  (random=0.0092  lift=+0.0066)
  R@10=0.0759  (random=0.0443)  n_eval=76

--- Fold 2/5  train=1352  val=337 ---
  train_rmse=0.2220  val_rmse=0.3284  best_val=0.3280 @epoch 57
  baseline=0.4531  improvement=27.5%
  P@10=0.0217  (random=0.0145  lift=+0.0072)
  R@10=0.0955  (random=0.0637)  n_eval=69

--- Fold 3/5  train=1352  val=337 ---
  train_rmse=0.2162  val_rmse=0.3536  best_val=0.3354 @epoch 10
  baseline=0.4451  improvement=20.6%
  P@10=0.0100  (random=0.0143  lift=-0.0043)
  R@10=0.0437  (random=0.0625)  n_eval=70

--- Fold 4/5  train=1352  val=337 ---
  train_rmse=0.2304  val_rmse=0.3441  best_val=0.3304 @epoch 8
  baseline=0.4433  improvement=22.4%
  P@10=0.0130  (random=0.0130  lift=+0.0000)
  R@10=0.0563  (random=0.0563)  n_eval=69

--- Fold 5/5  train=1348  val=341 ---
  train_rmse=0.2230  val_rmse=0.3472  best_val=0.3333 

In [81]:
results_df = (pd.DataFrame(results)
              .sort_values("mean_lift_p@10", ascending=False)
              .reset_index(drop=True))

print("\n========== All configs ranked by mean_lift_p@10 ==========")
print(results_df.to_string(index=False))

best = results_df.iloc[0]
print(f"\n>>> Best config (by lift_p@10):")
print(f"    factors={int(best['factors'])}  lr={best['lr']}  reg={best['reg']}  epochs={int(best['epochs'])}")
print(f"    mean P@10      = {best['mean_p@10']:.4f}  (lift {best['mean_lift_p@10']:+.4f})")
print(f"    mean Recall@10 = {best['mean_r@10']:.4f}")
print(f"    mean val_rmse  = {best['mean_val_rmse']:.4f}  ({best['mean_improvement_pct']:.1f}% over baseline)")

# best set of hyper-parameters
BEST_CONFIG = {
    "factors": int(best["factors"]),
    "lr":      float(best["lr"]),
    "reg":     float(best["reg"]),
    "epochs":  int(best["epochs"]),
}
print(f"\nBEST_CONFIG = {BEST_CONFIG}")


========== All configs ranked by mean_lift_p@10 ==========
 factors    lr  reg  epochs  mean_p@10  std_p@10  mean_lift_p@10  mean_r@10  mean_val_rmse  mean_improvement_pct
       4 0.050 0.03      60   0.014975  0.005806        0.003518   0.068314       0.368095             17.537788
       3 0.025 0.03      60   0.014970  0.006566        0.003514   0.068410       0.349507             21.700983
       3 0.030 0.05      60   0.014957  0.006147        0.003501   0.068434       0.347703             22.106011
       3 0.040 0.05      60   0.014775  0.005633        0.003318   0.067016       0.351828             21.180357
       3 0.030 0.03      60   0.014672  0.005864        0.003215   0.067160       0.352890             20.941921
       4 0.030 0.02      60   0.014536  0.008289        0.003080   0.065774       0.359522             19.461215
       3 0.040 0.03      60   0.014453  0.005423        0.002997   0.065806       0.360195             19.304098
       3 0.025 0.02      60   0.0144

In [84]:
# Refactoring best config after consulting the total table and the % improvement.
BEST_CONFIG = {"factors": 3, "lr": 0.025, "reg": 0.03, "epochs": 60}
print(f"\nBEST_CONFIG = {BEST_CONFIG}")


BEST_CONFIG = {'factors': 3, 'lr': 0.025, 'reg': 0.03, 'epochs': 60}
